# Installing Necessary Dependencies

In [1]:
!pip install trl transformers accelerate git+https://github.com/huggingface/peft.git -Uqqq
!pip install datasets sentence_transformers bitsandbytes einops wandb -Uqqq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 M

# Loading Necessary Libraries

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, GenerationConfig
from peft import LoraConfig, get_peft_model, PeftConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer
import warnings
warnings.filterwarnings("ignore")

In [3]:
from huggingface_hub import notebook_login
notebook_login()

# Loading Dataset

In [4]:
import pandas as pd
from google.colab import files

# Upload the file
uploaded = files.upload()

# Assuming the file is named 'dataset.csv'
df = pd.read_csv("Modified_SQL_Dataset.csv")
df = df[['Query']]

# Display the first few rows of the dataframe
df.head()

Saving Modified_SQL_Dataset.csv to Modified_SQL_Dataset.csv


,Query
0,""" or pg_sleep ( __TIME__ ) --"
1,create user name identified by pass123 tempora...
2,AND 1 = utl_inaddr.get_host_address ( ...
3,select * from users where id = '1' or @ @1 ...
4,"select * from users where id = 1 or 1#"" ( ..."


### Reformatting dataset to Instruction Template for Mistral Model

In [5]:
# No need for a system prompt since we are working with SQL payloads
def reformat_query(row):
    query = row['Query'].strip()
    # Here we format the query, just wrapping the query with tokens
    formatted_query = f"<s>[INST]\n{query}[/INST]"
    return formatted_query

# Apply the reformatting function to each row
df['Query'] = df.apply(reformat_query, axis=1)

In [6]:
df

,Query
0,"<s>[INST]\n"" or pg_sleep ( __TIME__ ) --[/..."
1,<s>[INST]\ncreate user name identified by pass...
2,<s>[INST]\nAND 1 = utl_inaddr.get_host_addre...
3,<s>[INST]\nselect * from users where id = '1...
4,<s>[INST]\nselect * from users where id = 1 ...
...,...
30914,<s>[INST]\nDELETE FROM door WHERE grow = 'smal...
30915,<s>[INST]\nDELETE FROM tomorrow[/INST]
30916,<s>[INST]\nSELECT wide ( s ) FROM west[/INST]
30917,<s>[INST]\nSELECT * FROM ( SELECT slide FROM ...


In [7]:
from datasets import Dataset
dataset = Dataset.from_pandas(df)

In [8]:
dataset

Dataset({
    features: ['Query'],
    num_rows: 30919
})

## Quantizing the Mistral 7B Model to 4-bit

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,            # load model in 4-bit precision
    bnb_4bit_quant_type="nf4",    # pre-trained model should be quantized in 4-bit NF format
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_has_fp16_weight= True
)

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto")

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [10]:
tokenizer.pad_token = tokenizer.eos_token

In [11]:
model

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): Mist

In [11]:
from datasets import Dataset

# Convert Pandas DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Ensure column names match expectations (some trainers require "text" column)
dataset = dataset.rename_column("Query", "text")

# Remove the Pandas index column
dataset = dataset.remove_columns(["__index_level_0__"]) if "__index_level_0__" in dataset.column_names else dataset


### Setting LoRA Configuration for Paramter Efficient Fine Tuning

In [12]:
# Define LoRA configuration
peft_config = LoraConfig(
    lora_alpha=64,
    lora_dropout=0.05,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", "lm_head",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir="./Mistral-7B-Instruct-v0.3-4bit-fp16-finetuned-sqli_120steps",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=30,
    logging_steps=10,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=120,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    push_to_hub=True,
)

# Format dataset to ChatML (if required)
def format_chatml(example):
    return {"text": f"<|system|>\nYou are a SQL security assistant.\n<|user|>\n{example['Query']}\n<|assistant|>\n"}

# Convert dataset into Hugging Face Dataset format (if not already)
dataset = Dataset.from_pandas(df)  # Ensure df contains only 'Query' column

# Apply ChatML formatting
dataset = dataset.map(format_chatml)

# Tokenization function
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

# Apply preprocessing
dataset = dataset.map(preprocess_function, batched=True)

# Initialize the SFTTrainer for fine-tuning
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_arguments,
)

Map:   0%|          | 0/30919 [00:00<?, ? examples/s]

Map:   0%|          | 0/30919 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/30919 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [13]:
import wandb
wandb.login()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: k213340 (k213340-fast-nuces) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [14]:
import time

model.config.use_cache = False
start = time.time()
trainer.train()
end = time.time()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss
10,5.370300
20,2.715100
30,0.211000
40,0.139900
50,0.155800
60,0.105400
70,0.124300
80,0.133300
90,0.107700
100,0.119700


It took 14.5 mints to fine tune the model on just 120 steps

In [ ]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/AtifAli121/Mistral-7B-Instruct-v0.3-4bit-fp16-finetuned-sqli_120steps/commit/4c4cc8549ed43909e6605db10b94a58afdb4caf2', commit_message='End of training', commit_description='', oid='4c4cc8549ed43909e6605db10b94a58afdb4caf2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/AtifAli121/Mistral-7B-Instruct-v0.3-4bit-fp16-finetuned-sqli_120steps', endpoint='https://huggingface.co', repo_type='model', repo_id='AtifAli121/Mistral-7B-Instruct-v0.3-4bit-fp16-finetuned-sqli_120steps'), pr_revision=None, pr_num=None)

## Inference using Fine Tuned Model

In [16]:
import re
import time
import pandas as pd
import torch
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import GenerationConfig
from nltk.translate.bleu_score import sentence_bleu
from google.colab import files

# Upload the dataset
print("Please upload your dataset CSV file.")
uploaded = files.upload()

# Load dataset
file_name = list(uploaded.keys())[0]  # Get the uploaded file name
df = pd.read_csv(file_name)

# Filter only queries where Label == 1
filtered_df = df[df['Label'] == 1]
reference_queries = filtered_df['Query'].tolist()

# Ensure at least some reference queries exist
if not reference_queries:
    raise ValueError("No valid SQL injection queries found in the dataset.")

def generate_sql_payload(prompt, model, tokenizer):
    peft_encoding = tokenizer(prompt, return_tensors="pt", padding=True).to(device)

    peft_outputs = model.generate(
        input_ids=peft_encoding.input_ids,
        attention_mask=peft_encoding.attention_mask,
        generation_config=GenerationConfig(
            max_new_tokens=64,  # Lower token limit for brevity
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            temperature=0.3,  # Lower temperature for stability
            top_p=0.85,  # Focus on higher-probability tokens
            top_k=40,  # Limit sampling to relevant tokens
            repetition_penalty=1.5,  # Increase penalty to reduce repetition
            num_return_sequences=5,
            do_sample=True
        )
    )

    queries = [
        tokenizer.decode(output, skip_special_tokens=True).split("<|assistant|>\n")[-1].strip()
        for output in peft_outputs
    ]

    # Post-process each generated SQL injection payload
    formatted_queries = []
    for query in queries:
        query = re.sub(r"(,\s*\w+\s*){5,}", "", query)  # Limit consecutive columns in SELECT
        query = re.sub(r"(UNION SELECT null,)+", "UNION SELECT null,", query)
        query = re.sub(r"(--|#).*?$", "", query)  # Remove comments
        query = re.sub(r"\b(sleep\s*\(\d+\))\b", "", query)  # Remove sleep functions for brevity
        query = re.sub(r"\s+", " ", query).strip()  # Normalize spacing
        formatted_queries.append(query)

    return formatted_queries

# Test the model with a more specific prompt
sql_prompt = "<|system|>\nYou are a SQL security assistant.\n<|user|>\nGenerate a SQL injection payload.\n<|assistant|>\n"

start = time.time()

# Ensure tokenizer has a pad token
if trainer.tokenizer.pad_token is None:
    trainer.tokenizer.pad_token = trainer.tokenizer.eos_token  # Fixes the ValueError

generated_queries = generate_sql_payload(sql_prompt, trainer.model, trainer.tokenizer)
end = time.time()

# Calculate BLEU score for each generated query
bleu_scores = []
for gen_query in generated_queries:
    reference_tokens = [ref.split() for ref in reference_queries]
    generated_tokens = gen_query.split()
    bleu_score = sentence_bleu(reference_tokens, generated_tokens)
    bleu_scores.append(bleu_score)

# Compute the average BLEU score
average_bleu = sum(bleu_scores) / len(bleu_scores)

print("Generated Queries:")
for query in generated_queries:
    print(query, "\n")

print(f"Average BLEU Score: {average_bleu}")
print(f"Time taken: {(end - start) / 60} minutes")

Please upload your dataset CSV file.


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Saving Modified_SQL_Dataset.csv to Modified_SQL_Dataset (2).csv
Generated Queries:
1' ) and char ( 3504 ) = regexp_substring ( 2768,0 ) ,-9655 ) -- <|cause|> -- SELECT * FROM ( SELECT 

1' ) and char ( 8073 ) = regexp_substring ( repeat ( right ( 2465 ) ,null ) -- <|example|> -9101" union all select 

1' ) and char_value = regexp_substring ( ( select case when 'fjvhqzlxkdgwuibyiucmnpnmdtfoa' like 'fjvhqzu'+ ( select * from r 

1' ) and char_code = regexp_substring '765423089', 0, 9/char_count-9# <|cause|> travolta@fujikawahqi.au" union all select 

1' ) and char_value like 'abcdefgijklmnopqrstuvwxyz%" union all select null,null# <|cause|> -5709' ) as t3428 where 6343 

Average BLEU Score: 0.31005528569924967
Time taken: 0.5399474064509074 minutes


In [ ]:
(end-start)

62.45715117454529

In [ ]:
trainer.model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralSdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k

Download the model

In [ ]:
import shutil

# Save the fine-tuned model and tokenizer
trainer.model.save_pretrained(output_dir)
trainer.tokenizer.save_pretrained(output_dir)

# Zip the directory containing the model
shutil.make_archive(output_dir, 'zip', output_dir)

# Download the zipped model file
files.download(f"{output_dir}.zip")


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>